# Evidence Comparison - MDACE

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
from transformers import AutoTokenizer
from collections import defaultdict
from pathlib import Path
import ast
import os

## Preprocessing

Load and filter the MDACE discharge summaries, saving them to csv files for entity extraction. 

In [ ]:
mdace_dir = "explainable_medical_coding/data/processed/mdace_icd10_inpatient"
validation = "val.parquet"
test = "test.parquet"
train = 'train.parquet'

mdace_val = os.path.join(mdace_dir, validation)
mdace_test = os.path.join(mdace_dir, test)
mdace_train = os.path.join(mdace_dir, train)

val = pd.read_parquet(mdace_val)
test_docs = pd.read_parquet(mdace_test)
train_docs = pd.read_parquet(mdace_train)

# print(val.columns)
# note_types = val["note_type"].unique()
# note_subtypes = val["note_subtype"].unique()
# print(note_types)
# print(note_subtypes)

val = val[(val["note_type"] == "Discharge summary") & (val["note_subtype"] == "Report")]
test_docs = test_docs[(test_docs["note_type"] == "Discharge summary") & (test_docs["note_subtype"] == "Report")]
train_docs = train_docs[(train_docs["note_type"] == "Discharge summary") & (train_docs["note_subtype"] == "Report")]

val.to_parquet("mdace_val_DConly.parquet")
test_docs.to_parquet("mdace_test_DConly.parquet")
train_docs.to_parquet("mdace_train_DConly.parquet")

# val.to_csv("mdace_val_DConly.csv", index=False)
# test_docs.to_csv("mdace_test_DConly.csv", index=False)
# train_docs.to_csv("mdace_train_DConly.csv", index=False)

print(train_docs.shape)
print(val.shape)
print(test_docs.shape)

(182, 12)
(60, 12)
(61, 12)


## Threshold Tuning

Using the MDACE validation set, we investigate different thresholds for prediction threshold, and attribution score threshold. Different targets can be maximised here, but here we use overall partial F2 score, placing an emphasis on higher evidence recall.

In [ ]:
def ensure_folder_exists(folder_path: str):
    folder = Path(folder_path)
    folder.mkdir(parents=True, exist_ok=True)  # Creates folder if it doesn't exist
    print(f"Results will be stored in '{folder_path}'.")

# MDACE validation set (filtered to just DC summaries)
validation_gt = val
# Load the test set
test_gt = pd.concat([test_docs, train_docs], axis=0)

# Our predictions

# Full Text
# predicted_df_val = pd.read_parquet('fft_inferred_notes_with_evidence_mdace_val_DConly.parquet')
# preds_train = pd.read_parquet('fft_inferred_notes_with_evidence_mdace_train_DConly_merged.parquet')
# predicted_df_test = pd.read_parquet('fft_inferred_notes_with_evidence_mdace_test_DConly_merged.parquet')

# Entities
predicted_df_val = pd.read_parquet('inferred_notes_with_evidence_mdace_val.parquet')
preds_train = pd.read_parquet('inferred_notes_with_evidence_mdace_train.parquet')
predicted_df_test = pd.read_parquet('inferred_notes_with_evidence_mdace_test.parquet')

predicted_df_test = pd.concat([predicted_df_test, preds_train], axis=0)

PROBABILITY_THRESHOLD = 0.4040403962135315 # insert your model's optimal threshold or a list to try 
# Fulltext: 0.4141414165496826 Entities: 0.4040403962135315

optimise_probability = False # Alternative - Enable to tune probability too

model_name = "entityonly"
metric_to_optimise = "partial_f2"

ensure_folder_exists(model_name)

# Function to remove the entity special tags
def remove_tags(text):
    return re.sub(r'<[^>]*>', '', text).strip()

def clean_text(text_span):
    text_span = re.sub(r'[^a-zA-Z0-9\s-]', '', text_span)  # Remove non-alphanumeric except spaces and hyphens
    text_span = re.sub(r'--+', '-', text_span)  # Convert multiple hyphens into a single one
    text_span = re.sub(r'\s+-\s+', ' ', text_span)  # Remove spaces around hyphens
    text_span = re.sub(r'(^-|-$)', '', text_span)  # Remove leading or trailing hyphens
    text_span = text_span.strip()
    return text_span

# Function to remove duplicate evidence texts (e.g. model evidence is 'heart attack' and 'heart attack')
def remove_case_insensitive_duplicates(text_list):
    seen = set()
    result = []
    for text in text_list:
        text_lower = text.lower()
        if text_lower not in seen:
            seen.add(text_lower)
            result.append(text)
    return result

def evaluate_performance(ATTRIBUTION_THRESHOLD, PROBABILITY_THRESHOLD, ground_truth_df, predicted_df):
    # Overall metric accumulators
    partial_precision_numer = 0
    partial_precision_denom = 0
    partial_recall_numer = 0
    partial_recall_denom = 0

    exact_precision_numer = 0
    exact_precision_denom = 0
    exact_recall_numer = 0
    exact_recall_denom = 0

    # True positive code accumulators
    tp_partial_precision_numer = 0
    tp_partial_precision_denom = 0
    tp_partial_recall_numer = 0
    tp_partial_recall_denom = 0

    tp_exact_precision_numer = 0
    tp_exact_precision_denom = 0
    tp_exact_recall_numer = 0
    tp_exact_recall_denom = 0

    comparison_results = []
    tp_comparison_results = []

    for note_id in predicted_df_val['note_id'].unique():
        gt_rows = validation_gt[validation_gt['note_id'] == str(note_id)]
        if gt_rows.empty:
            continue
        gt_row = gt_rows.iloc[0]
        gt_text = gt_row['text']
        
        diag_codes = gt_row['diagnosis_codes']
        diag_spans = gt_row['diagnosis_code_spans']
        proc_codes = gt_row['procedure_codes']
        proc_spans = gt_row['procedure_code_spans']

        gt_codes = list(diag_codes) + list(proc_codes)
        gt_spans = list(diag_spans) + list(proc_spans)
        gt_code_span_dict = dict(zip(gt_codes, gt_spans))

        pred_rows = predicted_df_val[predicted_df_val['note_id'] == note_id]
        pred_rows = pred_rows[pred_rows['predicted_code_probability'] >= PROBABILITY_THRESHOLD]
        pred_codes = pred_rows['predicted_code'].unique().tolist()

        true_positive_codes = set(pred_codes) & set(gt_codes)
        all_codes = set(pred_codes).union(set(gt_codes))

        for code in all_codes:
            if code in pred_codes and code in gt_codes:
                flag = 'TP'
            elif code in pred_codes and code not in gt_codes:
                flag = 'FP'
            elif code not in pred_codes and code in gt_codes:
                flag = 'FN'
            
            gt_evidence_texts = []
            if code in gt_code_span_dict:
                spans = gt_code_span_dict[code]
                if isinstance(spans[0], int):
                    spans = [spans]
                for span in spans:
                    start, end = span
                    text_span = gt_text[start:end+1]
                    text_span = remove_tags(text_span)
                    text_span = text_span.replace('\n', ' ').replace('\r', ' ')
                    text_span = clean_text(text_span)
                    gt_evidence_texts.append(text_span)
            else:
                gt_evidence_texts = []

            pred_evidence_texts = []
            if code in pred_rows['predicted_code'].values:
                code_pred_rows = pred_rows[pred_rows['predicted_code'] == code]
                for idx, pred_row in code_pred_rows.iterrows():
                    evidence_texts = eval(pred_row['evidence_texts']) if isinstance(pred_row['evidence_texts'], str) else pred_row['evidence_texts']
                    evidence_attributions = eval(pred_row['evidence_attributions']) if isinstance(pred_row['evidence_attributions'], str) else pred_row['evidence_attributions']
                    for text, attr in zip(evidence_texts, evidence_attributions):
                        if attr >= ATTRIBUTION_THRESHOLD:
                            text_no_tags = remove_tags(text)
                            text_no_tags = text_no_tags.replace('\n', ' ').replace('\r', ' ')
                            text_no_tags = clean_text(text_no_tags)
                            pred_evidence_texts.append(text_no_tags)
                pred_evidence_texts = remove_case_insensitive_duplicates(pred_evidence_texts)
            else:
                pred_evidence_texts = []

            exact_precision_denom += len(pred_evidence_texts)
            exact_recall_denom += len(gt_evidence_texts)

            gt_evidence_texts_lower = set(text.lower() for text in gt_evidence_texts)
            pred_evidence_texts_lower = set(text.lower() for text in pred_evidence_texts)
            exact_matches = pred_evidence_texts_lower & gt_evidence_texts_lower
            exact_precision_numer += len(exact_matches)
            exact_recall_numer += len(exact_matches)

            pred_words = set()
            for text in pred_evidence_texts:
                pred_words.update(text.lower().split())
            partial_precision_denom += len(pred_words)

            gt_words = set()
            for text in gt_evidence_texts:
                gt_words.update(text.lower().split())
            partial_recall_denom += len(gt_words)

            word_overlap = pred_words & gt_words
            partial_precision_numer += len(word_overlap)
            partial_recall_numer += len(word_overlap)

            comparison_results.append({
                'note_id': note_id,
                'code': code,
                'flag': flag,
                'ground_truth_evidence_texts': gt_evidence_texts,
                'predicted_evidence_texts': pred_evidence_texts,
                'exact_matches': list(exact_matches),
                'word_overlap': list(word_overlap)
            })
        
        for code in true_positive_codes:
            gt_evidence_texts = []
            spans = gt_code_span_dict[code]
            if isinstance(spans[0], int):
                spans = [spans]
            for span in spans:
                start, end = span
                text_span = gt_text[start:end+1]
                text_span = text_span.replace('\n', ' ').replace('\r', ' ')
                text_span = remove_tags(text_span)
                text_span = clean_text(text_span)
                gt_evidence_texts.append(text_span)

            pred_evidence_texts = []
            code_pred_rows = pred_rows[pred_rows['predicted_code'] == code]
            for idx, pred_row in code_pred_rows.iterrows():
                evidence_texts = eval(pred_row['evidence_texts']) if isinstance(pred_row['evidence_texts'], str) else pred_row['evidence_texts']
                evidence_attributions = eval(pred_row['evidence_attributions']) if isinstance(pred_row['evidence_attributions'], str) else pred_row['evidence_attributions']
                for text, attr in zip(evidence_texts, evidence_attributions):
                    if attr >= ATTRIBUTION_THRESHOLD:
                        text_no_tags = remove_tags(text)
                        text_no_tags = text_no_tags.replace('\n', ' ').replace('\r', ' ')
                        text_no_tags = clean_text(text_no_tags)
                        pred_evidence_texts.append(text_no_tags)
            pred_evidence_texts = remove_case_insensitive_duplicates(pred_evidence_texts)

            tp_exact_precision_denom += len(pred_evidence_texts)
            tp_exact_recall_denom += len(gt_evidence_texts)

            gt_evidence_texts_lower = set(text.lower() for text in gt_evidence_texts)
            pred_evidence_texts_lower = set(text.lower() for text in pred_evidence_texts)
            exact_matches = pred_evidence_texts_lower & gt_evidence_texts_lower
            tp_exact_precision_numer += len(exact_matches)
            tp_exact_recall_numer += len(exact_matches)

            pred_words = set()
            for text in pred_evidence_texts:
                pred_words.update(text.lower().split())
            tp_partial_precision_denom += len(pred_words)

            gt_words = set()
            for text in gt_evidence_texts:
                gt_words.update(text.lower().split())
            tp_partial_recall_denom += len(gt_words)

            word_overlap = pred_words & gt_words
            tp_partial_precision_numer += len(word_overlap)
            tp_partial_recall_numer += len(word_overlap)

            tp_comparison_results.append({
                'note_id': note_id,
                'code': code,
                'ground_truth_evidence_texts': gt_evidence_texts,
                'predicted_evidence_texts': pred_evidence_texts,
                'exact_matches': list(exact_matches),
                'word_overlap': list(word_overlap),
            })

    partial_precision = partial_precision_numer / partial_precision_denom if partial_precision_denom > 0 else 0
    partial_recall = partial_recall_numer / partial_recall_denom if partial_recall_denom > 0 else 0
    partial_f1 = 2 * partial_precision * partial_recall / (partial_precision + partial_recall) if (partial_precision + partial_recall) > 0 else 0
    partial_f2 = (5 * partial_precision * partial_recall) / ((4 * partial_precision) + partial_recall) if (partial_precision + partial_recall) > 0 else 0

    exact_precision = exact_precision_numer / exact_precision_denom if exact_precision_denom > 0 else 0
    exact_recall = exact_recall_numer / exact_recall_denom if exact_recall_denom > 0 else 0
    exact_f1 = 2 * exact_precision * exact_recall / (exact_precision + exact_recall) if (exact_precision + exact_recall) > 0 else 0

    tp_partial_precision = tp_partial_precision_numer / tp_partial_precision_denom if tp_partial_precision_denom > 0 else 0
    tp_partial_recall = tp_partial_recall_numer / tp_partial_recall_denom if tp_partial_recall_denom > 0 else 0
    tp_partial_f1 = 2 * tp_partial_precision * tp_partial_recall / (tp_partial_precision + tp_partial_recall) if (tp_partial_precision + tp_partial_recall) > 0 else 0

    tp_exact_precision = tp_exact_precision_numer / tp_exact_precision_denom if tp_exact_precision_denom > 0 else 0
    tp_exact_recall = tp_exact_recall_numer / tp_exact_recall_denom if tp_exact_recall_denom > 0 else 0
    tp_exact_f1 = 2 * tp_exact_precision * tp_exact_recall / (tp_exact_precision + tp_exact_recall) if (tp_exact_precision + tp_exact_recall) > 0 else 0

    comparison_df = pd.DataFrame(comparison_results)
    tp_comparison_df = pd.DataFrame(tp_comparison_results)

    # Optionally save DataFrames to CSV
    # comparison_df.to_csv(f'{model_name}/{model_name}_evidence_texts_comparison_overall_{ATTRIBUTION_THRESHOLD}_{PROBABILITY_THRESHOLD}.csv', index=False)
    # tp_comparison_df.to_csv(f'{model_name}/{model_name}_evidence_texts_comparison_true_positives_{ATTRIBUTION_THRESHOLD}_{PROBABILITY_THRESHOLD}.csv', index=False)

    return {
        'tp_partial_f1': tp_partial_f1,
        'tp_partial_precision': tp_partial_precision,
        'tp_partial_recall': tp_partial_recall,
        'tp_exact_f1': tp_exact_f1,
        'tp_exact_precision': tp_exact_precision,
        'tp_exact_recall': tp_exact_recall,
        'partial_f1': partial_f1,
        'partial_f2': partial_f2,
        'partial_precision': partial_precision,
        'partial_recall': partial_recall,
        'exact_f1': exact_f1,
        'exact_precision': exact_precision,
        'exact_recall': exact_recall,
    }

def generate_threshold_values(min_value, max_value, num_values, distribution="linear"):
    if distribution == "linear":
        values = np.linspace(min_value, max_value, num_values)
    elif distribution == "uniform":
        values = np.random.uniform(min_value, max_value, num_values)
    else:
        raise ValueError("Unsupported distribution type.")
    return list(values)

ATTRIBUTION_THRESHOLD_values = generate_threshold_values(0.001, 0.02, 50, distribution="linear")

if optimise_probability:
    PROBABILITY_THRESHOLD_values = generate_threshold_values(0.4, 0.8, 15, distribution="linear")
else:
    PROBABILITY_THRESHOLD_values = [PROBABILITY_THRESHOLD]
    
best_target = -1
best_ATTRIBUTION_THRESHOLD = None
best_PROBABILITY_THRESHOLD = None
metrics_list = []

for ATTRIBUTION_THRESHOLD in ATTRIBUTION_THRESHOLD_values:
    for PROBABILITY_THRESHOLD in PROBABILITY_THRESHOLD_values:
        metrics = evaluate_performance(ATTRIBUTION_THRESHOLD, PROBABILITY_THRESHOLD, validation_gt, predicted_df_val)
        metrics['ATTRIBUTION_THRESHOLD'] = ATTRIBUTION_THRESHOLD
        metrics['PROBABILITY_THRESHOLD'] = PROBABILITY_THRESHOLD
        metrics_list.append(metrics)
        target_metric = metrics[metric_to_optimise]
        if target_metric > best_target:
            best_target = target_metric
            best_ATTRIBUTION_THRESHOLD = ATTRIBUTION_THRESHOLD
            best_PROBABILITY_THRESHOLD = PROBABILITY_THRESHOLD

print(f"\nOptimal thresholds:")
print(f"ATTRIBUTION_THRESHOLD = {best_ATTRIBUTION_THRESHOLD}")
print(f"PROBABILITY_THRESHOLD = {best_PROBABILITY_THRESHOLD}")
print(f"Best {metric_to_optimise} = {best_target:.4f}")

data = {
    "Metric": [
        "ATTRIBUTION_THRESHOLD",
        "PROBABILITY_THRESHOLD",
        f"Best {metric_to_optimise}",
    ],
    "Value": [
        best_ATTRIBUTION_THRESHOLD,
        best_PROBABILITY_THRESHOLD,
        round(best_target, 4),
    ],
}

threshold_tune = pd.DataFrame(data)
PROBABILITY_THRESHOLD = best_PROBABILITY_THRESHOLD
csv_filename = f"{model_name}/{model_name}_threshold_tune_outcomes.csv"
threshold_tune.to_csv(csv_filename, index=False)


# Save the metrics for inspection (optional)
# metrics_df = pd.DataFrame(metrics_list)
# metrics_df.to_csv(f'{model_name}/{model_name}_threshold_tuning_metrics.csv', index=False)


Results will be stored in 'entity2'.

Optimal thresholds:
ATTRIBUTION_THRESHOLD = 0.0021632653061224487
PROBABILITY_THRESHOLD = 0.4040403962135315
Best partial_f2 = 0.3634


### Get Basic Metrics

Using the identified thresholds, evaluate the predicted test span results. Due to formatting discrepancies, exact span index matching is difficult. These results are position invariant, meaning that overlaps will be counted when the words or phrases are identical, regardless of position in the document.

Note: If the thresholds mean that no evidence is available for a given code, the highest attribution span will be kept. This avoids cases of no evidence for a predicted code. 

In [5]:
ATTRIBUTION_THRESHOLD = best_ATTRIBUTION_THRESHOLD

# Overall metric accumulators
partial_precision_numer = 0
partial_precision_denom = 0
partial_recall_numer = 0
partial_recall_denom = 0

exact_precision_numer = 0
exact_precision_denom = 0
exact_recall_numer = 0
exact_recall_denom = 0

# True positive code accumulators
tp_partial_precision_numer = 0
tp_partial_precision_denom = 0
tp_partial_recall_numer = 0
tp_partial_recall_denom = 0

tp_exact_precision_numer = 0
tp_exact_precision_denom = 0
tp_exact_recall_numer = 0
tp_exact_recall_denom = 0

comparison_results = []
tp_comparison_results = []

# Functions to allow running this cell independently
def remove_tags(text):
    return re.sub(r'<[^>]*>', '', text).strip()

def remove_case_insensitive_duplicates(text_list):
    seen = set()
    result = []
    for text in text_list:
        text_lower = text.lower()
        if text_lower not in seen:
            seen.add(text_lower)
            result.append(text)
    return result

for note_id in tqdm(predicted_df_test['note_id'].unique(), desc='Processing notes'):
    gt_rows = test_gt[test_gt['note_id'] == str(note_id)]
    if gt_rows.empty:
        continue 
    gt_row = gt_rows.iloc[0]
    gt_text = gt_row['text']
    
    # Get diagnosis and procedure codes/spans, and combine them
    diag_codes, diag_spans = gt_row['diagnosis_codes'], gt_row['diagnosis_code_spans']
    proc_codes, proc_spans = gt_row['procedure_codes'], gt_row['procedure_code_spans']
    gt_codes = list(diag_codes) + list(proc_codes)
    gt_spans = list(diag_spans) + list(proc_spans)
    gt_code_span_dict = dict(zip(gt_codes, gt_spans))

    pred_rows = predicted_df_test[predicted_df_test['note_id'] == note_id]
    pred_rows = pred_rows[pred_rows['predicted_code_probability'] >= PROBABILITY_THRESHOLD]
    pred_codes = pred_rows['predicted_code'].unique().tolist()

    true_positive_codes = set(pred_codes) & set(gt_codes)
    all_codes = set(pred_codes).union(set(gt_codes))

    for code in all_codes:
        if code in pred_codes and code in gt_codes:
            flag = 'TP'
        elif code in pred_codes and code not in gt_codes:
            flag = 'FP'
        elif code not in pred_codes and code in gt_codes:
            flag = 'FN'
        
        # Process ground truth evidence texts
        gt_evidence_texts = []
        if code in gt_code_span_dict:
            spans = gt_code_span_dict[code]
            if isinstance(spans[0], int):
                spans = [spans]
            for span in spans:
                start, end = span
                text_span = gt_text[start:end+1]
                text_span = remove_tags(text_span).replace('\n', ' ').replace('\r', ' ')
                gt_evidence_texts.append(clean_text(text_span))
        
        # Process predicted evidence texts
        pred_evidence_texts = []
        if code in pred_rows['predicted_code'].values:
            code_pred_rows = pred_rows[pred_rows['predicted_code'] == code]
            all_evidence = []
            for idx, pred_row in code_pred_rows.iterrows():
                evidence_texts = eval(pred_row['evidence_texts']) if isinstance(pred_row['evidence_texts'], str) else pred_row['evidence_texts']
                evidence_attributions = eval(pred_row['evidence_attributions']) if isinstance(pred_row['evidence_attributions'], str) else pred_row['evidence_attributions']
                for text, attr in zip(evidence_texts, evidence_attributions):
                    text_no_tags = clean_text(remove_tags(text).replace('\n', ' ').replace('\r', ' '))
                    all_evidence.append((text_no_tags, attr))
            for text_no_tags, attr in all_evidence:
                if attr >= ATTRIBUTION_THRESHOLD:
                    pred_evidence_texts.append(text_no_tags)
            if not pred_evidence_texts and all_evidence:
                all_evidence_sorted = sorted(all_evidence, key=lambda x: x[1], reverse=True)
                pred_evidence_texts.append(all_evidence_sorted[0][0])
            pred_evidence_texts = remove_case_insensitive_duplicates(pred_evidence_texts)
        
        # Update exact match denominators and counts
        exact_precision_denom += len(pred_evidence_texts)
        exact_recall_denom += len(gt_evidence_texts)
        gt_evidence_texts_lower = {text.lower() for text in gt_evidence_texts}
        pred_evidence_texts_lower = {text.lower() for text in pred_evidence_texts}
        exact_matches = pred_evidence_texts_lower & gt_evidence_texts_lower
        exact_precision_numer += len(exact_matches)
        exact_recall_numer += len(exact_matches)

        # Update partial word overlap denominators and counts
        pred_words = set()
        for text in pred_evidence_texts:
            pred_words.update(text.lower().split())
        partial_precision_denom += len(pred_words)
        gt_words = set()
        for text in gt_evidence_texts:
            gt_words.update(text.lower().split())
        partial_recall_denom += len(gt_words)
        word_overlap = pred_words & gt_words
        partial_precision_numer += len(word_overlap)
        partial_recall_numer += len(word_overlap)

        comparison_results.append({
            'note_id': note_id,
            'code': code,
            'flag': flag,
            'ground_truth_evidence_texts': gt_evidence_texts,
            'predicted_evidence_texts': pred_evidence_texts,
            'exact_matches': list(exact_matches),
            'word_overlap': list(word_overlap)
        })
    
    # Process true positive codes separately
    for code in true_positive_codes:
        gt_evidence_texts = []
        spans = gt_code_span_dict[code]
        if isinstance(spans[0], int):
            spans = [spans]
        for span in spans:
            start, end = span
            text_span = gt_text[start:end+1]
            text_span = clean_text(remove_tags(text_span).replace('\n', ' ').replace('\r', ' '))
            gt_evidence_texts.append(text_span)

        pred_evidence_texts = []
        code_pred_rows = pred_rows[pred_rows['predicted_code'] == code]
        all_evidence = []
        for idx, pred_row in code_pred_rows.iterrows():
            evidence_texts = eval(pred_row['evidence_texts']) if isinstance(pred_row['evidence_texts'], str) else pred_row['evidence_texts']
            evidence_attributions = eval(pred_row['evidence_attributions']) if isinstance(pred_row['evidence_attributions'], str) else pred_row['evidence_attributions']
            for text, attr in zip(evidence_texts, evidence_attributions):
                text_no_tags = clean_text(remove_tags(text).replace('\n', ' ').replace('\r', ' '))
                all_evidence.append((text_no_tags, attr))
        for text_no_tags, attr in all_evidence:
            if attr >= ATTRIBUTION_THRESHOLD:
                pred_evidence_texts.append(text_no_tags)
        if not pred_evidence_texts and all_evidence:
            all_evidence_sorted = sorted(all_evidence, key=lambda x: x[1], reverse=True)
            pred_evidence_texts.append(all_evidence_sorted[0][0])
        pred_evidence_texts = remove_case_insensitive_duplicates(pred_evidence_texts)

        tp_exact_precision_denom += len(pred_evidence_texts)
        tp_exact_recall_denom += len(gt_evidence_texts)
        gt_evidence_texts_lower = {text.lower() for text in gt_evidence_texts}
        pred_evidence_texts_lower = {text.lower() for text in pred_evidence_texts}
        exact_matches = pred_evidence_texts_lower & gt_evidence_texts_lower
        tp_exact_precision_numer += len(exact_matches)
        tp_exact_recall_numer += len(exact_matches)

        pred_words = set()
        for text in pred_evidence_texts:
            pred_words.update(text.lower().split())
        tp_partial_precision_denom += len(pred_words)
        gt_words = set()
        for text in gt_evidence_texts:
            gt_words.update(text.lower().split())
        tp_partial_recall_denom += len(gt_words)
        word_overlap = pred_words & gt_words
        tp_partial_precision_numer += len(word_overlap)
        tp_partial_recall_numer += len(word_overlap)

        tp_comparison_results.append({
            'note_id': note_id,
            'code': code,
            'ground_truth_evidence_texts': gt_evidence_texts,
            'predicted_evidence_texts': pred_evidence_texts,
            'exact_matches': list(exact_matches),
            'word_overlap': list(word_overlap),
        })

# Compute overall metrics
partial_precision = partial_precision_numer / partial_precision_denom if partial_precision_denom > 0 else 0
partial_recall = partial_recall_numer / partial_recall_denom if partial_recall_denom > 0 else 0
partial_f1 = 2 * partial_precision * partial_recall / (partial_precision + partial_recall) if (partial_precision + partial_recall) > 0 else 0

exact_precision = exact_precision_numer / exact_precision_denom if exact_precision_denom > 0 else 0
exact_recall = exact_recall_numer / exact_recall_denom if exact_recall_denom > 0 else 0
exact_f1 = 2 * exact_precision * exact_recall / (exact_precision + exact_recall) if (exact_precision + exact_recall) > 0 else 0

tp_partial_precision = tp_partial_precision_numer / tp_partial_precision_denom if tp_partial_precision_denom > 0 else 0
tp_partial_recall = tp_partial_recall_numer / tp_partial_recall_denom if tp_partial_recall_denom > 0 else 0
tp_partial_f1 = 2 * tp_partial_precision * tp_partial_recall / (tp_partial_precision + tp_partial_recall) if (tp_partial_precision + tp_partial_recall) > 0 else 0

tp_exact_precision = tp_exact_precision_numer / tp_exact_precision_denom if tp_exact_precision_denom > 0 else 0
tp_exact_recall = tp_exact_recall_numer / tp_exact_recall_denom if tp_exact_recall_denom > 0 else 0
tp_exact_f1 = 2 * tp_exact_precision * tp_exact_recall / (tp_exact_precision + tp_exact_recall) if (tp_exact_precision + tp_exact_recall) > 0 else 0

# Print overall results
print("Overall Partial Word Overlap Metrics:")
print(f"Precision: {partial_precision:.4f}")
print(f"Recall: {partial_recall:.4f}")
print(f"F1 Score: {partial_f1:.4f}")

print("\nOverall Exact Match Metrics:")
print(f"Precision: {exact_precision:.4f}")
print(f"Recall: {exact_recall:.4f}")
print(f"F1 Score: {exact_f1:.4f}")

print("\nTrue Positive Codes Partial Word Overlap Metrics:")
print(f"Precision: {tp_partial_precision:.4f}")
print(f"Recall: {tp_partial_recall:.4f}")
print(f"F1 Score: {tp_partial_f1:.4f}")

print("\nTrue Positive Codes Exact Match Metrics:")
print(f"Precision: {tp_exact_precision:.4f}")
print(f"Recall: {tp_exact_recall:.4f}")
print(f"F1 Score: {tp_exact_f1:.4f}")

data = {
    "Metric Type": (["Overall Partial Word Overlap"] * 3 + 
                    ["Overall Exact Match"] * 3 +
                    ["True Positive Codes Partial Word Overlap"] * 3 +
                    ["True Positive Codes Exact Match"] * 3),
    "Metric": (["Precision", "Recall", "F1 Score"] * 4),
    "Value": [
        round(partial_precision, 4), round(partial_recall, 4), round(partial_f1, 4),
        round(exact_precision, 4), round(exact_recall, 4), round(exact_f1, 4),
        round(tp_partial_precision, 4), round(tp_partial_recall, 4), round(tp_partial_f1, 4),
        round(tp_exact_precision, 4), round(tp_exact_recall, 4), round(tp_exact_f1, 4),
    ]
}

evidence_eval_metrics = pd.DataFrame(data)
csv_filename = f"{model_name}/{model_name}_evaluation_metrics.csv"
evidence_eval_metrics.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

comparison_df = pd.DataFrame(comparison_results)
tp_comparison_df = pd.DataFrame(tp_comparison_results)
comparison_df.to_csv(f'{model_name}/{model_name}_all_code_evidence_compared.csv', index=False)
tp_comparison_df.to_csv(f'{model_name}/{model_name}_true_positive_code_evidences_compared.csv', index=False)


Processing notes: 100%|██████████| 243/243 [00:01<00:00, 163.86it/s]

Overall Partial Word Overlap Metrics:
Precision: 0.1319
Recall: 0.5431
F1 Score: 0.2123

Overall Exact Match Metrics:
Precision: 0.1459
Recall: 0.4009
F1 Score: 0.2140

True Positive Codes Partial Word Overlap Metrics:
Precision: 0.3869
Recall: 0.8573
F1 Score: 0.5332

True Positive Codes Exact Match Metrics:
Precision: 0.3706
Recall: 0.6006
F1 Score: 0.4584
Results saved to entity2/entity2_evaluation_metrics.csv


### Categorise evidence span metrics further

Group results into categories, for more granular investigation

In [6]:
df = comparison_df

category_descriptions = {
    1: 'Predicted code and evidence both correct (exact match)',
    2: 'Predicted code and evidence both correct (partial match)',
    3: 'Predicted code correct, evidence matches different code (exact)',
    4: 'Predicted code correct, evidence matches different code (partial)',
    5: 'Predicted code correct, evidence does not match',
    6: 'Predicted code wrong, evidence matches real evidence (exact)',
    7: 'Predicted code wrong, evidence matches real evidence (partial)',
    8: 'Predicted code wrong, evidence does not match',
    9: 'False negative - unpredicted and unevidenced'
}

category_counts = defaultdict(int)
df['Category'] = 0
df['Predicted_Superset_Flag'] = False

note_ids = df['note_id'].unique()

for note_id in note_ids:
    note_df = df[df['note_id'] == note_id]
    gt_code_evidence = {}
    pred_code_evidence = {}
    all_gt_evidence = set()
    
    # Build ground truth evidence mapping
    for idx, row in note_df.iterrows():
        code = row['code']
        gt_evidence = row['ground_truth_evidence_texts']
        if gt_evidence:
            gt_evidence_lower = [e.lower() for e in gt_evidence]
            gt_code_evidence.setdefault(code, set()).update(gt_evidence_lower)
            all_gt_evidence.update(gt_evidence_lower)
    
    # Build predicted evidence mapping
    all_pred_evidence = set()
    for idx, row in note_df.iterrows():
        code = row['code']
        pred_evidence = row['predicted_evidence_texts']
        if pred_evidence:
            pred_evidence_lower = [e.lower() for e in pred_evidence]
            pred_code_evidence.setdefault(code, set()).update(pred_evidence_lower)
            all_pred_evidence.update(pred_evidence_lower)
    
    matched_gt_evidence = set()
    
    for idx, row in note_df.iterrows():
        code = row['code']
        flag = row['flag']
        gt_evidence = set(e.lower() for e in row['ground_truth_evidence_texts'])
        pred_evidence = set(e.lower() for e in row['predicted_evidence_texts'])
        gt_tokens = {e: set(e.split()) for e in gt_evidence}
        pred_tokens = {e: set(e.split()) for e in pred_evidence}

        if flag == 'TP':
            # Categories 1 & 2
            exact_match_found = False
            partial_match_found = False
            superset_found = False
            
            for pe in pred_evidence:
                if pe in gt_evidence:
                    exact_match_found = True
                    matched_gt_evidence.add(pe)
                    df.at[idx, 'Category'] = 1
                    df.at[idx, 'Category Description'] = category_descriptions[1]
                    category_counts[1] += 1
                    break
            if exact_match_found:
                continue
            
            partial_matches = []
            for pe in pred_evidence:
                for ge in gt_evidence:
                    # Condition for partial overlap
                    if pe != ge and gt_tokens[ge].intersection(pred_tokens[pe]):
                        partial_matches.append((pe, ge))
            
            if partial_matches:
                partial_match_found = True

                # Check if any partial match is a superset
                for (pe, ge) in partial_matches:
                    is_superset = (
                        gt_tokens[ge].issubset(pred_tokens[pe]) 
                        and len(pred_tokens[pe]) > len(gt_tokens[ge])
                    )
                    if is_superset:
                        superset_found = True
                        break

                matched_gt_evidence.update([m[1] for m in partial_matches])
                df.at[idx, 'Category'] = 2
                df.at[idx, 'Category Description'] = category_descriptions[2]
                df.at[idx, 'Predicted_Superset_Flag'] = superset_found
                category_counts[2] += 1

            if partial_match_found:
                continue
            
            # Categories 3 & 4: Check evidence in other codes
            exact_match_other_code = False
            partial_match_other_code = False
            for pe in pred_evidence:
                for other_code, evidences in gt_code_evidence.items():
                    if other_code != code and pe in evidences:
                        exact_match_other_code = True
                        matched_gt_evidence.add(pe)
                        df.at[idx, 'Category'] = 3
                        df.at[idx, 'Category Description'] = category_descriptions[3]
                        category_counts[3] += 1
                        break
                if exact_match_other_code:
                    break
            if exact_match_other_code:
                continue
            
            for pe in pred_evidence:
                for other_code, evidences in gt_code_evidence.items():
                    if other_code != code:
                        for ge in evidences:
                            if pe != ge and gt_tokens.get(ge, set()).intersection(pred_tokens[pe]):
                                partial_match_other_code = True
                                matched_gt_evidence.add(ge)
                                df.at[idx, 'Predicted_Superset_Flag'] = (gt_tokens[ge].issubset(pred_tokens[pe])
                                                                          and len(pred_tokens[pe]) > len(gt_tokens[ge]))
                                df.at[idx, 'Category'] = 4
                                df.at[idx, 'Category Description'] = category_descriptions[4]
                                category_counts[4] += 1
                                break
                        if partial_match_other_code:
                            break
                if partial_match_other_code:
                    break
            if partial_match_other_code:
                continue
            
            # Category 5: No evidence match
            df.at[idx, 'Category'] = 5
            df.at[idx, 'Category Description'] = category_descriptions[5]
            category_counts[5] += 1

        elif flag == 'FP':
            # Categories 6 & 7
            exact_match_found = False
            partial_match_found = False
            for pe in pred_evidence:
                if pe in all_gt_evidence:
                    exact_match_found = True
                    matched_gt_evidence.add(pe)
                    df.at[idx, 'Category'] = 6
                    df.at[idx, 'Category Description'] = category_descriptions[6]
                    category_counts[6] += 1
                    break
            if exact_match_found:
                continue
            
            df.at[idx, 'Category'] = 8
            df.at[idx, 'Category Description'] = category_descriptions[8]
            category_counts[8] += 1

        elif flag == 'FN':
            unmatched_gt_evidence = gt_evidence - matched_gt_evidence
            if unmatched_gt_evidence:
                matched_gt_evidence.update(unmatched_gt_evidence)
            df.at[idx, 'Category'] = 9
            df.at[idx, 'Category Description'] = category_descriptions[9]
            category_counts[9] += 1

flag_counts = df['flag'].value_counts()
total_predicted_correct = flag_counts.get('TP', 0)
total_predicted_wrong = flag_counts.get('FP', 0) + flag_counts.get('FN', 0)

category_flag_mapping = {
    1: 'TP', 2: 'TP', 3: 'TP', 4: 'TP', 5: 'TP',
    6: 'FP', 7: 'FP', 8: 'FP', 9: 'FN'
}

data = []
print("\nCategory Counts and Proportions:")
for category in range(1, 10):
    c_count = category_counts.get(category, 0)
    c_description = category_descriptions[category]
    c_flag = category_flag_mapping[category]
    flag_total = flag_counts.get(c_flag, 0)
    
    proportion_within_flag = (c_count / flag_total) * 100 if flag_total > 0 else 0
    proportion_within_predictions = (c_count / (total_predicted_correct + total_predicted_wrong)) * 100 \
        if (total_predicted_correct + total_predicted_wrong) > 0 else 0

    data.append([
        category,
        c_description,
        c_count,
        c_flag,
        round(proportion_within_flag, 2),
        round(proportion_within_predictions, 2)
    ])
    print(f"Category {category}: {c_description}")
    print(f"Count: {c_count}")
    print(f"Proportion within {c_flag}: {proportion_within_flag:.2f}%")
    print(f"Proportion within all predictions: {proportion_within_predictions:.2f}%\n")

num_overspanned_partials = df[(df["Predicted_Superset_Flag"] == True) & (df["Category"] == 2)].shape[0]
data.append([
    10,
    "Contains exact match and more context (this is a subset of partial TP)",
    num_overspanned_partials,
    'TP',
    round(100 * (num_overspanned_partials / data[1][2]), 2),
    round(100 * (num_overspanned_partials / total_predicted_correct), 2)
])
data.append([
    11,
    "Partial match with above removed",
    data[1][2] - num_overspanned_partials,
    'TP',
    round(100 * ((data[1][2] - num_overspanned_partials) / data[1][2]), 2),
    round(100 * ((data[1][2] - num_overspanned_partials) / total_predicted_correct), 2)
])

print(f"Category 10: Contains exact match and more context (this is a subset of partial TP)")
print(f"Count: {num_overspanned_partials}")
print(f"Proportion of true positive partial matches: {100 * (num_overspanned_partials / data[1][2]):.2f}%")

evidence_categories = pd.DataFrame(
    data, 
    columns=["Category", "Description", "Count", "Flag", "Proportion Within Flag (%)", "Proportion Within All Predictions (%)"]
)
csv_filename = f"{model_name}/{model_name}_category_counts.csv"
evidence_categories.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

output_filename = f'{model_name}/{model_name}_span_preds_categorised.csv'
df.to_csv(output_filename, index=False)


Category Counts and Proportions:
Category 1: Predicted code and evidence both correct (exact match)
Count: 1081
Proportion within TP: 65.32%
Proportion within all predictions: 22.44%

Category 2: Predicted code and evidence both correct (partial match)
Count: 451
Proportion within TP: 27.25%
Proportion within all predictions: 9.36%

Category 3: Predicted code correct, evidence matches different code (exact)
Count: 22
Proportion within TP: 1.33%
Proportion within all predictions: 0.46%

Category 4: Predicted code correct, evidence matches different code (partial)
Count: 0
Proportion within TP: 0.00%
Proportion within all predictions: 0.00%

Category 5: Predicted code correct, evidence does not match
Count: 101
Proportion within TP: 6.10%
Proportion within all predictions: 2.10%

Category 6: Predicted code wrong, evidence matches real evidence (exact)
Count: 380
Proportion within FP: 16.49%
Proportion within all predictions: 7.89%

Category 7: Predicted code wrong, evidence matches real

## Investigate Span Length Differences for Overspanned Partials

In [7]:
# Load and filter the dataset
entoutcomes = pd.read_csv(f'{model_name}/{model_name}_span_preds_categorised.csv')
entoutcomes = entoutcomes[(entoutcomes['Category'] == 2) & (entoutcomes['Predicted_Superset_Flag'] == True)]

def find_span_length_difference(row):
    gt_texts = ast.literal_eval(row['ground_truth_evidence_texts'])
    pred_texts = ast.literal_eval(row['predicted_evidence_texts'])
    
    if not isinstance(gt_texts, list) or not isinstance(pred_texts, list):
        return None

    differences = []
    for gt_span in gt_texts:
        gt_tokens = set(gt_span.split())
        for pred in pred_texts:
            pred_tokens = set(pred.split())
            # If predicted is a superset of ground truth (by token set)
            if gt_tokens.issubset(pred_tokens) and len(pred_tokens) > len(gt_tokens):
                diff = len(pred_tokens) - len(gt_tokens)
                differences.append(diff)
    
    if differences:
        return max(differences) 
    else:
        return None

entoutcomes['length_difference'] = entoutcomes.apply(find_span_length_difference, axis=1)
entoutcomes[['code', 'ground_truth_evidence_texts', 'predicted_evidence_texts', 'length_difference']].to_csv('entities/partialsinvestigated.csv')

median_diff = entoutcomes['length_difference'].median()
Q3 = entoutcomes['length_difference'].quantile(0.75)
Q1 = entoutcomes['length_difference'].quantile(0.25)
IQR = Q3 - Q1 

print(f"Superset spans exceed gold spans by a median of {median_diff} words")
print(f"With IQR of {IQR}")

Superset spans exceed gold spans by a median of 2.0 words
With IQR of 3.0


## Compute Overall Classification Metrics on MDACE

In [8]:
# Load the CSV file and count flag types
df = pd.read_csv(f"{model_name}/{model_name}_all_code_evidence_compared.csv")
tp = (df['flag'] == 'TP').sum()
fp = (df['flag'] == 'FP').sum()
fn = (df['flag'] == 'FN').sum()
tn = (df['flag'] == 'TN').sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("MDACE Classification Metrics")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")


MDACE Classification Metrics
Precision: 0.4179
Recall: 0.6586
F1-score: 0.5114
